In [1]:
!pip install transformers peft datasets torch torchvision --quiet

In [2]:
from transformers import CLIPModel, CLIPProcessor
import torch

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

print("Model loaded!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Model loaded!
Total parameters: 151,277,313


In [3]:
from PIL import Image
import requests

# Load any image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# Zero-shot classification
texts = ["a photo of a cat", "a photo of a dog", "a photo of a car"]
inputs = processor(text=texts, images=image, return_tensors="pt", padding=True)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits_per_image
    probs = logits.softmax(dim=1)

for text, prob in zip(texts, probs[0]):
    print(f"{text}: {prob:.3f}")

a photo of a cat: 0.993
a photo of a dog: 0.005
a photo of a car: 0.002


In [4]:
from peft import LoraConfig, get_peft_model

# Configure LoRA
lora_config = LoraConfig(
    r=4,                    # rank — same as her paper
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],  # apply to attention layers
    lora_dropout=0.1,
    bias="none"
)

# Apply LoRA to the model
lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()
# This will show something like:
# trainable params: 294,912 || all params: 151,277,056 || trainable%: 0.19

trainable params: 245,760 || all params: 151,523,073 || trainable%: 0.1622


In [5]:
from datasets import load_dataset

# Load a simple image classification dataset
dataset = load_dataset("from torch.utils.data import DataLoader
from transformers import CLIPProcessor
import torch.nn as nn

# Class names for CIFAR10
class_names = ["airplane", "automobile", "bird", "cat", "deer", 
               "dog", "frog", "horse", "ship", "truck"]

# Create text prompts for each class
text_prompts = [f"a photo of a {name}" for name in class_names]

optimizer = torch.optim.AdamW(lora_model.parameters(), lr=1e-4)
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_model = lora_model.to(device)

# Simple training loop
lora_model.train()
for epoch in range(3):
    total_loss = 0
    for i, example in enumerate(dataset):
        if i >= 100:  # just 100 steps to test
            break
            
        image = example["img"]
        label = example["label"]
        
        # Process inputs
        inputs = processor(
            text=text_prompts,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)
        
        outputs = lora_model(**inputs)
        logits = outputs.logits_per_image  # shape: [1, 10]
        
        # Loss: correct class should have highest similarity
        target = torch.tensor([label]).to(device)
        loss = nn.CrossEntropyLoss()(logits, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Average Loss: {total_loss/100:.4f}")

print("Fine-tuning done!")", split="train[:1000]")  # just 1000 examples to start
print(dataset)
print(dataset[0])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/23.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Dataset({
    features: ['img', 'label'],
    num_rows: 1000
})
{'img': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=32x32 at 0x7D72A17ED250>, 'label': 0}


In [6]:
from torch.utils.data import DataLoader
from transformers import CLIPProcessor
import torch.nn as nn

# Class names for CIFAR10
class_names = ["airplane", "automobile", "bird", "cat", "deer", 
               "dog", "frog", "horse", "ship", "truck"]

# Create text prompts for each class
text_prompts = [f"a photo of a {name}" for name in class_names]

optimizer = torch.optim.AdamW(lora_model.parameters(), lr=1e-4)
device = "cuda" if torch.cuda.is_available() else "cpu"
lora_model = lora_model.to(device)

# Simple training loop
lora_model.train()
for epoch in range(3):
    total_loss = 0
    for i, example in enumerate(dataset):
        if i >= 100:  # just 100 steps to test
            break
            
        image = example["img"]
        label = example["label"]
        
        # Process inputs
        inputs = processor(
            text=text_prompts,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)
        
        outputs = lora_model(**inputs)
        logits = outputs.logits_per_image  # shape: [1, 10]
        
        # Loss: correct class should have highest similarity
        target = torch.tensor([label]).to(device)
        loss = nn.CrossEntropyLoss()(logits, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Average Loss: {total_loss/100:.4f}")

print("Fine-tuning done!")

Epoch 1, Average Loss: 0.3415
Epoch 2, Average Loss: 0.0756
Epoch 3, Average Loss: 0.0122
Fine-tuning done!


In [8]:
lora_model.eval()
correct = 0
total = 0

test_dataset = load_dataset("cifar10", split="test[:200]")

with torch.no_grad():
    for example in test_dataset:
        image = example["img"]
        label = example["label"]
        
        inputs = processor(
            text=text_prompts,
            images=image,
            return_tensors="pt",
            padding=True
        ).to(device)
        
        outputs = lora_model(**inputs)
        predicted = outputs.logits_per_image.argmax().item()
        
        if predicted == label:
            correct += 1
        total += 1

print(f"Accuracy after fine-tuning: {correct/total*100:.1f}%")

Accuracy after fine-tuning: 93.0%
